# Mel + Chroma

## Tanítás

In [ ]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from gensim.models import Word2Vec

# --- ÚTVONALAK A COLABON ---
H5_PATH = "/content/spotify_dataset_compressed.h5"
W2V_PATH = "/content/song2vec.model"
# ÚJ MENTÉSI ÚTVONAL: Hogy ne írjuk felül a régi (3 ágú) modelledet!
SAVE_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_cnn_notempo.keras"

# ==========================================
# 1. A PINCÉR: DATA GENERATOR (TEMPO NÉLKÜL)
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64, shuffle=True):
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_indices = np.sort(batch_indices)

        with h5py.File(self.h5_path, 'r') as hf:
            X_mel = hf['spectrograms/mel'][batch_indices]
            X_chroma = hf['spectrograms/chroma'][batch_indices]
            # X_tempo TÖRÖLVE!
            uris = hf['ml/track_uri'][batch_indices]

        # Csatorna dimenzió hozzáadása
        X_mel = np.expand_dims(X_mel, axis=-1)
        X_chroma = np.expand_dims(X_chroma, axis=-1)

        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            if uri in self.w2v_model.wv:
                y_batch[i] = self.w2v_model.wv[uri]

        # VISSZATÉRÉS CSAK 2 BEMENETTEL!
        return (X_mel, X_chroma), y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

# ==========================================
# 2. AZ AGY: MODELL ARCHITEKTÚRA (TEMPO NÉLKÜL)
# ==========================================
def create_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

print("\n🚀 Modell építése...")
input_mel    = Input(shape=(128, 1280, 1), name='mel_input')
input_chroma = Input(shape=(12,  1280, 1), name='chroma_input')
# input_tempo TÖRÖLVE!

branch_mel = create_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")
branch_chroma = create_branch(input_chroma, [16, 32, 64], [(1,4), (1,4), (1,4)], "chroma")
# branch_tempo TÖRÖLVE!

# ÖSSZEFŰZÉS CSAK 2 ÁGGAL!
merged = Concatenate(name='concat_all_branches')([branch_mel, branch_chroma])

z = Dense(1024, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)
z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.4, name='dropout_2')(z)

output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = L2NormLayer(name='l2_norm')(output_raw)

# CSAK 2 BEMENET!
model = Model(inputs=[input_mel, input_chroma], outputs=output_layer)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=cosine_loss, metrics=['mae'])

# ==========================================
# 3. A STARTPISZTOLY: TANÍTÁS
# ==========================================
print("\n📚 Word2Vec betöltése...")
w2v_model = Word2Vec.load(W2V_PATH)

print("\n⚙️ Generátorok inicializálása...")
train_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='train', batch_size=64)
val_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='val', batch_size=64, shuffle=False)

if os.path.exists(SAVE_PATH):
    print(f"\n💾 Korábbi állapot megtalálva!")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    print("✅ Modell sikeresen betöltve!")
else:
    print(f"\n⚠️ Tiszta lappal indulunk! Nincs mentés ezen a néven: {SAVE_PATH}")

callbacks = [
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    # ÚJ: Ha 3 körig nem javul a val_loss, lefelezi a tanulási rátát!
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)
]

print("\n🔥 GPU TANÍTÁS INDÍTÁSA (TEMPO NÉLKÜL)...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15, # Nyugodtan mehet magasra, az EarlyStopping leállítja
    callbacks=callbacks
)


🚀 Modell építése...

📚 Word2Vec betöltése...

⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

⚠️ Tiszta lappal indulunk! Nincs mentés ezen a néven: /content/drive/MyDrive/Diplomamunka/spotify_cnn_notempo.keras

🔥 GPU TANÍTÁS INDÍTÁSA (TEMPO NÉLKÜL)...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.6804 - mae: 0.1604
Epoch 1: val_loss improved from None to 0.59713, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_notempo.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_notempo.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 787s 2s/step - loss: 0.5602 - mae: 0.1557 - val_loss: 0.5971 - val_mae: 0.1573 - learning_rate: 0.0010
Epoch 2/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.4416 - mae: 0.1506
Epoch 2: val_loss improved from 0.59713 to 0.54215, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_notempo.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_notempo.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 743s 2s/step - loss: 0.4355 - mae: 0.1503 - val_loss: 0.5421 - val_mae: 0.1550 - learning_rate: 0.0010
Epoch 3/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.4194 - mae: 0.1494
Epoch 3: val_loss did not improve from 0.5421

## Kiértékelés

### Releváns uri, word2vec térben keresés

In [1]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_notempo.keras" 
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*60)
    print("VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity)")
    print("="*60)
    
    print(f"\nGLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)    AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN MODELL -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
    print("\n" + "="*60)

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:12<00:00, 501605.72it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 1000 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 1000/1000 [05:45<00:00,  2.89it/s]



VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity)

GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.43%
  CNN MODEL (Audio)    AUC: 70.28%

--- TOP 1 AJÁNLÁS ---
  BASELINE -> HR: 91.30% | Prec: 91.30% | Rec: 0.02%
  CNN MODELL -> HR: 15.80% | Prec: 15.80% | Rec: 0.00%

--- TOP 5 AJÁNLÁS ---
  BASELINE -> HR: 97.50% | Prec: 86.94% | Rec: 0.07%
  CNN MODELL -> HR: 35.70% | Prec: 14.96% | Rec: 0.01%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 84.41% | Rec: 0.13%
  CNN MODELL -> HR: 47.10% | Prec: 14.69% | Rec: 0.01%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 99.30% | Prec: 80.93% | Rec: 0.24%
  CNN MODELL -> HR: 56.40% | Prec: 14.58% | Rec: 0.02%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.46% | Rec: 0.53%
  CNN MODELL -> HR: 68.10% | Prec: 14.23% | Rec: 0.05%

--- TOP 100 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 70.46% | Rec: 0.93%
  CNN MODELL -> HR: 76.60% | Prec: 14.19% | Rec: 0.11%

--- TOP 200 AJÁNLÁS ---
  BASELINE -> HR: 99.90% | Prec: 64

### Releváns uri, hibrid téren (valós környezet)

In [2]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score
import copy  # Hozzáadva a hibrid modell klónozásához!

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_notempo.keras" 
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix_notempo.npy" # A létrehozott hibrid mátrixod

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n[+] HIBRID vektortér betöltése a memóriába...")
    # Klónozzuk a w2v modellt, hogy a szótár (vocab) megmaradjon
    hybrid_w2v = copy.deepcopy(w2v_model)
    
    # Betöltjük a numpy mátrixot
    hybrid_matrix = np.load(HYBRID_MATRIX_PATH)
    
    # FELÜLÍRJUK a klónozott modell vektorait a hibrid mátrixszal!
    hybrid_w2v.wv.vectors = hybrid_matrix
    
    # KRITIKUS LÉPÉS: Újraszámoljuk a normákat a cosine similarity-hez
    hybrid_w2v.wv.fill_norms(force=True) 
    print("    Hibrid vektortér sikeresen integrálva!")

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár (kibővítve a hibriddel)
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "hybrid":   {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- C) HIBRID MODELL ---
            hybrid_similars = hybrid_w2v.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            hybrid_recs = [u for u, score in hybrid_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))
            results["hybrid"]["auc"].append(calculate_auc(pred_vector, relevant_uris, hybrid_w2v))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)
                
                # Hybrid
                hr_h, prec_h, rec_h = calculate_metrics(hybrid_recs, relevant_uris, k)
                results["hybrid"][k]["hr"].append(hr_h)
                results["hybrid"][k]["prec"].append(prec_h)
                results["hybrid"][k]["rec"].append(rec_h)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*70)
    print("VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆")
    print("="*70)
    
    print(f"\nGLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)   AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")
    print(f"  HIBRID MODELL       AUC: {np.mean(results['hybrid']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN      -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
        print(f"  HIBRID   -> HR: {np.mean(results['hybrid'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['hybrid'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['hybrid'][k]['rec'])*100:.2f}%")
    print("\n" + "="*70)

if __name__ == "__main__":
    main()

1. Modellek betöltése...

[+] HIBRID vektortér betöltése a memóriába...
    Hibrid vektortér sikeresen integrálva!

2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:13<00:00, 496527.66it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 1000 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 1000/1000 [07:53<00:00,  2.11it/s]



VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆

GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.43%
  CNN MODEL (Audio)   AUC: 70.30%
  HIBRID MODELL       AUC: 56.23%

--- TOP 1 AJÁNLÁS ---
  BASELINE -> HR: 91.30% | Prec: 91.30% | Rec: 0.02%
  CNN      -> HR: 15.80% | Prec: 15.80% | Rec: 0.00%
  HIBRID   -> HR: 55.70% | Prec: 55.70% | Rec: 0.01%

--- TOP 5 AJÁNLÁS ---
  BASELINE -> HR: 97.50% | Prec: 86.94% | Rec: 0.07%
  CNN      -> HR: 35.70% | Prec: 14.96% | Rec: 0.01%
  HIBRID   -> HR: 87.50% | Prec: 55.62% | Rec: 0.03%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 84.41% | Rec: 0.13%
  CNN      -> HR: 47.10% | Prec: 14.69% | Rec: 0.01%
  HIBRID   -> HR: 92.20% | Prec: 55.07% | Rec: 0.06%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 99.30% | Prec: 80.93% | Rec: 0.24%
  CNN      -> HR: 56.40% | Prec: 14.58% | Rec: 0.02%
  HIBRID   -> HR: 96.20% | Prec: 54.02% | Rec: 0.12%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.46% | Rec: 0.53%
  CNN    

### Szekvenciális hibrid téren

In [8]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix_notempo.npy"
TEST_PIDS_PATH = "../Models/test_pids.npy"

# Az új K értékek listája
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) # A FAISS kereséshez csak a legnagyobbra lesz szükségünk

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés előkészítése
# Egy szótárat (dictionary) hozunk létre, ami minden K értékhez tárolja a metrikákat
results = {k: {"hr_count": 0, "map_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
total_playlists = 0

print(f"Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists[::10]): # Maradt a minden 10. lista a gyorsasághoz
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = pl[:-1]
    target_idx = pl[-1]
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Most a MAX_K-ra (500) + a kontextus hosszára keresünk
    D, I = index.search(centroid_norm, MAX_K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == MAX_K:
            break

    # --- METRIKÁK SZÁMÍTÁSA MINDEN K-RA ---
    total_playlists += 1
    
    if target_idx in recommendations:
        # Megkeressük, hányadik helyen van a találat a max 500-as listában
        rank = recommendations.index(target_idx)
        
        # Végigmegyünk a K értékeken, és frissítjük azokat, amikbe belefér a találat
        for k in K_VALUES:
            if rank < k:
                results[k]["hr_count"] += 1
                results[k]["map_sum"] += calculate_map(rank, k)
                results[k]["ndcg_sum"] += calculate_ndcg(rank, k)

# 4. Eredmények összesítése és kiíratása
print("\n" + "="*50)
print("VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)")
print("="*50)

for k in K_VALUES:
    final_hr = results[k]["hr_count"] / total_playlists
    final_map = results[k]["map_sum"] / total_playlists
    final_ndcg = results[k]["ndcg_sum"] / total_playlists
    
    print(f"\n--- TOP {k} AJÁNLÁS ---")
    print(f"  Hit Rate (HR@{k}):  {final_hr*100:.2f}%")
    print(f"  MAP@{k}:            {final_map*100:.4f}%")
    print(f"  NDCG@{k}:           {final_ndcg*100:.4f}%")

print("\n" + "="*50)

Adatok betöltése...
Vektorok normalizálása...
FAISS index építése...
Kiértékelés futtatása 99,180 listán...


100%|██████████| 9918/9918 [05:46<00:00, 28.59it/s]


VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)

--- TOP 1 AJÁNLÁS ---
  Hit Rate (HR@1):  0.30%
  MAP@1:            0.3025%
  NDCG@1:           0.3025%

--- TOP 5 AJÁNLÁS ---
  Hit Rate (HR@5):  0.70%
  MAP@5:            0.4393%
  NDCG@5:           0.5026%

--- TOP 10 AJÁNLÁS ---
  Hit Rate (HR@10):  1.08%
  MAP@10:            0.4901%
  NDCG@10:           0.6262%

--- TOP 20 AJÁNLÁS ---
  Hit Rate (HR@20):  1.69%
  MAP@20:            0.5312%
  NDCG@20:           0.7797%

--- TOP 50 AJÁNLÁS ---
  Hit Rate (HR@50):  3.14%
  MAP@50:            0.5732%
  NDCG@50:           1.0590%

--- TOP 100 AJÁNLÁS ---
  Hit Rate (HR@100):  5.24%
  MAP@100:            0.6023%
  NDCG@100:           1.3981%

--- TOP 200 AJÁNLÁS ---
  Hit Rate (HR@200):  8.39%
  MAP@200:            0.6241%
  NDCG@200:           1.8348%

--- TOP 500 AJÁNLÁS ---
  Hit Rate (HR@500):  14.88%
  MAP@500:            0.6447%
  NDCG@500:           2.6141%



### Szekvenciális, tartalom alapú

In [1]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/ae_vectors_closed_world_notempo.npy"
TEST_PIDS_PATH = "../Models/test_pids_ae.npy"

# Az új K értékek listája
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) # A FAISS kereséshez csak a legnagyobbra lesz szükségünk

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés előkészítése
# Egy szótárat (dictionary) hozunk létre, ami minden K értékhez tárolja a metrikákat
results = {k: {"hr_count": 0, "map_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
total_playlists = 0

print(f"Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists): # Maradt a minden 10. lista a gyorsasághoz
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = [idx - 1 for idx in pl[:-1]]
    target_idx = pl[-1] - 1
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Most a MAX_K-ra (500) + a kontextus hosszára keresünk
    D, I = index.search(centroid_norm, MAX_K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == MAX_K:
            break

    # --- METRIKÁK SZÁMÍTÁSA MINDEN K-RA ---
    total_playlists += 1
    
    if target_idx in recommendations:
        # Megkeressük, hányadik helyen van a találat a max 500-as listában
        rank = recommendations.index(target_idx)
        
        # Végigmegyünk a K értékeken, és frissítjük azokat, amikbe belefér a találat
        for k in K_VALUES:
            if rank < k:
                results[k]["hr_count"] += 1
                results[k]["map_sum"] += calculate_map(rank, k)
                results[k]["ndcg_sum"] += calculate_ndcg(rank, k)

# 4. Eredmények összesítése és kiíratása
print("\n" + "="*50)
print("VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)")
print("="*50)

for k in K_VALUES:
    final_hr = results[k]["hr_count"] / total_playlists
    final_map = results[k]["map_sum"] / total_playlists
    final_ndcg = results[k]["ndcg_sum"] / total_playlists
    
    print(f"\n--- TOP {k} AJÁNLÁS ---")
    print(f"  Hit Rate (HR@{k}):  {final_hr*100:.2f}%")
    print(f"  MAP@{k}:            {final_map*100:.4f}%")
    print(f"  NDCG@{k}:           {final_ndcg*100:.4f}%")

print("\n" + "="*50)

Adatok betöltése...
Vektorok normalizálása...
FAISS index építése...
Kiértékelés futtatása 88,316 listán...


100%|██████████| 88316/88316 [05:15<00:00, 279.70it/s]


VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)

--- TOP 1 AJÁNLÁS ---
  Hit Rate (HR@1):  0.02%
  MAP@1:            0.0238%
  NDCG@1:           0.0238%

--- TOP 5 AJÁNLÁS ---
  Hit Rate (HR@5):  0.10%
  MAP@5:            0.0488%
  NDCG@5:           0.0613%

--- TOP 10 AJÁNLÁS ---
  Hit Rate (HR@10):  0.18%
  MAP@10:            0.0586%
  NDCG@10:           0.0855%

--- TOP 20 AJÁNLÁS ---
  Hit Rate (HR@20):  0.32%
  MAP@20:            0.0680%
  NDCG@20:           0.1208%

--- TOP 50 AJÁNLÁS ---
  Hit Rate (HR@50):  0.71%
  MAP@50:            0.0801%
  NDCG@50:           0.1983%

--- TOP 100 AJÁNLÁS ---
  Hit Rate (HR@100):  1.33%
  MAP@100:            0.0886%
  NDCG@100:           0.2982%

--- TOP 200 AJÁNLÁS ---
  Hit Rate (HR@200):  2.45%
  MAP@200:            0.0964%
  NDCG@200:           0.4540%

--- TOP 500 AJÁNLÁS ---
  Hit Rate (HR@500):  5.33%
  MAP@500:            0.1054%
  NDCG@500:           0.7979%



### Seed

In [2]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---

HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix_notempo.npy"
TEST_PIDS_PATH = "../Models/test_pids.npy"

# K értékek és a seed méretek definiálása
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) 
SEED_LENGTHS = list(range(1, 11)) # 1-től 10-ig

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')

# CSAK MINDEN 100. LISTA BETÖLTÉSE A [::100] SZELETELÉSSEL
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)[::100]
print(f"Kiértékelendő listák száma a szűrés után: {len(test_playlists)}")

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés futtatása minden seed méretre
for seed_len in SEED_LENGTHS:
    print("\n" + "━"*70)
    print(f"🎯 EREDMÉNYEK: {seed_len} MEGADOTT DAL (SEED) ESETÉN")
    print("━"*70)

    # Metrikák tárolóinak inicializálása az adott seed mérethez
    results = {k: {"recall_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
    total_clicks = 0
    total_r_prec = 0.0
    total_playlists = 0

    for pl in tqdm(test_playlists, desc=f"Seed {seed_len} kiértékelése"):
        # Ha a lista nem hosszabb a seed méreténél, nincs mit megprediktálni
        if len(pl) <= seed_len:
            continue
        
        # 0-s indexelés miatt kivonunk 1-et az ID-kből
        context_indices = [idx - 1 for idx in pl[:seed_len]]
        target_indices = set([idx - 1 for idx in pl[seed_len:]])
        num_targets = len(target_indices)
        
        # --- SÚLYOZOTT CENTROID KÉPZÉSE (csak a seed dalokon) ---
        vectors = hybrid_matrix[context_indices]
        weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
        
        weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
        
        # Keresés előtti normalizálás
        centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
        centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

        # --- KERESÉS (FAISS) ---
        # Keresünk annyit, hogy a leválogatás után is maradjon MAX_K (500) elem
        D, I = index.search(centroid_norm, MAX_K + len(context_indices))
        
        # Ajánlások szűrése (ne ajánljuk azt, ami a seed-ben már benne van)
        recommendations = []
        for idx in I[0]:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == MAX_K:
                break

        # --- METRIKÁK SZÁMÍTÁSA ---
        total_playlists += 1
        
        # Megkeressük, hányadik helyen vannak a találatok (1-es alapú indexeléssel a képletekhez)
        hit_ranks = []
        for rank_0_idx, rec in enumerate(recommendations):
            if rec in target_indices:
                hit_ranks.append(rank_0_idx + 1)
        
        # 1. Clicks (Hány 10-es lapozás kell az első találathoz)
        if len(hit_ranks) > 0:
            first_hit = hit_ranks[0]
            clicks = (first_hit - 1) // 10
        else:
            clicks = 51 # Ha nincs találat az 500-ban, büntetés
        total_clicks += clicks

        # 2. R-Precision
        hits_in_r = sum(1 for r in hit_ranks if r <= num_targets)
        total_r_prec += (hits_in_r / num_targets)

        # 3. Recall és NDCG minden K-ra
        for k in K_VALUES:
            k_hits = [r for r in hit_ranks if r <= k]
            
            # Recall@K
            results[k]['recall_sum'] += len(k_hits) / num_targets
            
            # NDCG@K
            dcg = sum(1.0 / np.log2(r + 1) for r in k_hits)
            
            idcg_len = min(k, num_targets)
            idcg = sum(1.0 / np.log2(pos + 1) for pos in range(1, idcg_len + 1))
            
            if idcg > 0:
                results[k]['ndcg_sum'] += (dcg / idcg)

    # 4. Eredmények összesítése az adott seed méretre
    if total_playlists == 0:
        print("Nincs elegendő lejátszási lista ehhez a seed mérethez a teszthalmazban.")
        continue

    print(f"\n--- Kiértékelve {total_playlists:,} listán ---")
    
    # K-szintű metrikák formázott kiíratása
    print(f"{'K':<5} | {'Recall':<10} | {'NDCG':<10}")
    print("-" * 35)
    for k in K_VALUES:
        final_recall = results[k]["recall_sum"] / total_playlists
        final_ndcg = results[k]["ndcg_sum"] / total_playlists
        print(f"{k:<5} | {final_recall:.4f}   | {final_ndcg:.4f}")

    # Általános metrikák
    avg_clicks = total_clicks / total_playlists
    avg_r_prec = total_r_prec / total_playlists

    print(f"\n🎯 R-Precision: {avg_r_prec:.4f}")
    print(f"🖱️ Clicks:      {avg_clicks:.4f}\n")

Adatok betöltése...
Kiértékelendő listák száma a szűrés után: 992
Vektorok normalizálása...
FAISS index építése...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 EREDMÉNYEK: 1 MEGADOTT DAL (SEED) ESETÉN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Seed 1 kiértékelése: 100%|██████████| 992/992 [00:36<00:00, 26.98it/s]



--- Kiértékelve 992 listán ---
K     | Recall     | NDCG      
-----------------------------------
1     | 0.0000   | 0.0010
5     | 0.0002   | 0.0015
10    | 0.0003   | 0.0014
20    | 0.0006   | 0.0016
50    | 0.0013   | 0.0017
100   | 0.0025   | 0.0021
200   | 0.0053   | 0.0034
500   | 0.0128   | 0.0067

🎯 R-Precision: 0.0014
🖱️ Clicks:      39.9415


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 EREDMÉNYEK: 2 MEGADOTT DAL (SEED) ESETÉN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Seed 2 kiértékelése: 100%|██████████| 992/992 [00:36<00:00, 26.81it/s]



--- Kiértékelve 992 listán ---
K     | Recall     | NDCG      
-----------------------------------
1     | 0.0000   | 0.0020
5     | 0.0003   | 0.0022
10    | 0.0005   | 0.0021
20    | 0.0007   | 0.0020
50    | 0.0016   | 0.0022
100   | 0.0032   | 0.0029
200   | 0.0062   | 0.0041
500   | 0.0139   | 0.0074

🎯 R-Precision: 0.0020
🖱️ Clicks:      38.0464


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 EREDMÉNYEK: 3 MEGADOTT DAL (SEED) ESETÉN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


...


--- Kiértékelve 956 listán ---
K     | Recall     | NDCG      
-----------------------------------
1     | 0.0000   | 0.0010
5     | 0.0003   | 0.0025
10    | 0.0006   | 0.0025
20    | 0.0010   | 0.0024
50    | 0.0024   | 0.0028
100   | 0.0046   | 0.0035
200   | 0.0077   | 0.0050
500   | 0.0168   | 0.0090

🎯 R-Precision: 0.0021
🖱️ Clicks:      35.8860


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 EREDMÉNYEK: 9 MEGADOTT DAL (SEED) ESETÉN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Seed 9 kiértékelése: 100%|██████████| 992/992 [00:34<00:00, 28.36it/s]



--- Kiértékelve 947 listán ---
K     | Recall     | NDCG      
-----------------------------------
1     | 0.0000   | 0.0032
5     | 0.0003   | 0.0027
10    | 0.0006   | 0.0028
20    | 0.0015   | 0.0030
50    | 0.0037   | 0.0035
100   | 0.0053   | 0.0040
200   | 0.0099   | 0.0056
500   | 0.0197   | 0.0097

🎯 R-Precision: 0.0023
🖱️ Clicks:      35.7856


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 EREDMÉNYEK: 10 MEGADOTT DAL (SEED) ESETÉN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Seed 10 kiértékelése: 100%|██████████| 992/992 [00:34<00:00, 28.70it/s]


--- Kiértékelve 928 listán ---
K     | Recall     | NDCG      
-----------------------------------
1     | 0.0000   | 0.0032
5     | 0.0004   | 0.0032
10    | 0.0007   | 0.0032
20    | 0.0010   | 0.0027
50    | 0.0022   | 0.0030
100   | 0.0059   | 0.0039
200   | 0.0096   | 0.0055
500   | 0.0190   | 0.0097

🎯 R-Precision: 0.0025
🖱️ Clicks:      35.2705

